# Creating dataset for the Hunting Leaking(missing values)project 

In [2]:
# 1. Create a simulated Admissions dataset with missing values (None/NaN)

import pandas as pd
import sqlite3
import numpy as np

admission_data = {
    "admission_id":[8001, 8002, 8003, 8004, 8005, 8006, 8007, 8008],
    "patient_name":["Baby Doe", "John Elder", "Jane Doe", "Kid Smith", "Old Man Logan", "Adult Brown", "Teen Green", "Senior White"],
    "department":["Pediatrics", "Geriatrics", "Cardiology", "Pediatrics", "Geriatrics", "Cardiology", "Pediatrics", "Geriatrics"],
    "age":[2, 85, 45, None, 90, 38, 14, None],
    "admission_weight_kg":[None, 70, 82, 15, None, 95, 55, 68]   
}
# add the dataset to the DataFrame
df_admission_data = pd.DataFrame(admission_data)
# connect the sql
connt = sqlite3.connect(":memory:")
# save the data to sqlite3
df_admission_data.to_sql("admissions", connt, index=False, if_exists="replace")
# create a function for query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("***************** Data has been created!!! ******************")

***************** Data has been created!!! ******************


# Finding the Null values

In [9]:
# query for all data

all_data = "SELECT * FROM admissions"
print("******************************** table for all data ****************************")
display(run_query(all_data))

# query for null/NaN values
admission_leaks = """
SELECT * FROM admissions WHERE department IN ('Pediatrics','Geriatrics') 
AND (age IS NULL OR admission_weight_kg IS NULL)

"""
print("/////////////////////////////////////////////////////////////////////")
print("******************* leaking values ******************************")
display(run_query(admission_leaks))

******************************** table for all data ****************************


,admission_id,patient_name,department,age,admission_weight_kg
0,8001,Baby Doe,Pediatrics,2.0,NaN
1,8002,John Elder,Geriatrics,85.0,70.0
2,8003,Jane Doe,Cardiology,45.0,82.0
3,8004,Kid Smith,Pediatrics,NaN,15.0
4,8005,Old Man Logan,Geriatrics,90.0,NaN
5,8006,Adult Brown,Cardiology,38.0,95.0
6,8007,Teen Green,Pediatrics,14.0,55.0
7,8008,Senior White,Geriatrics,NaN,68.0


/////////////////////////////////////////////////////////////////////
******************* leaking values ******************************


,admission_id,patient_name,department,age,admission_weight_kg
0,8001,Baby Doe,Pediatrics,2.0,NaN
1,8004,Kid Smith,Pediatrics,NaN,15.0
2,8005,Old Man Logan,Geriatrics,90.0,NaN
3,8008,Senior White,Geriatrics,NaN,68.0


# Query for count of missing ages per department

In [16]:
# query for missing ages per department
admission_missing_age = """
SELECT department, patient_name, COUNT(*) 
FROM admissions 
WHERE age IS NULL
GROUP BY department, patient_name
"""
print("************************* missing in age section ***************")
display(run_query(admission_missing_age))

************************* missing in age section ***************


,department,patient_name,COUNT(*)
0,Geriatrics,Senior White,1
1,Pediatrics,Kid Smith,1
